# Interpreting Detachment Results

This notebook demonstrates how to interpret detachment efficiency
results from Helicon simulations. We cover:

1. The three definitions of detachment efficiency
2. Field line topology classification
3. Plume divergence and beam efficiency
4. Electron magnetization mapping
5. Pressure anisotropy diagnostics

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from helicon.fields import Coil, Grid, compute_bfield
from helicon.fields.field_lines import (
    FieldLineType,
    compute_flux_function,
    trace_field_lines,
)

## 1. Setup: Compute the magnetic field

We start with a simple converging-diverging nozzle configuration.

In [ ]:
coils = [
    Coil(z=0.0, r=0.10, I=50000),  # throat coil
    Coil(z=-0.3, r=0.15, I=20000),  # upstream mirror
]
grid = Grid(z_min=-0.5, z_max=3.0, r_max=0.6, nz=256, nr=128)

bfield = compute_bfield(coils, grid, backend="numpy")
print(f"B_z range: [{bfield.Bz.min():.4f}, {bfield.Bz.max():.4f}] T")
print(f"B_r range: [{bfield.Br.min():.4f}, {bfield.Br.max():.4f}] T")

## 2. Field line topology

Trace field lines and classify them as open, closed, or separatrix.
Only particles on **open** field lines contribute to net thrust.

In [ ]:
fls = trace_field_lines(bfield, n_lines=25, z_start=0.0)

fig, ax = plt.subplots(figsize=(12, 4))
colors = {
    FieldLineType.OPEN: "tab:blue",
    FieldLineType.CLOSED: "tab:red",
    FieldLineType.SEPARATRIX: "tab:orange",
}
for line in fls.lines:
    ax.plot(line.z, line.r, color=colors[line.line_type], alpha=0.6, lw=0.8)

# Legend
for lt, c in colors.items():
    ax.plot([], [], color=c, label=lt.value)
ax.legend()
ax.set_xlabel("z [m]")
ax.set_ylabel("r [m]")
ax.set_title("Field line topology")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 3. Magnetic flux function

The flux function $\psi(r,z) = \int_0^r B_z(r', z) \, r' \, dr'$ defines
field line topology. Contours of $\psi$ are poloidal field lines.

In [ ]:
psi = compute_flux_function(bfield)

fig, ax = plt.subplots(figsize=(12, 4))
R, Z = np.meshgrid(bfield.r, bfield.z, indexing="ij")
levels = np.linspace(psi.min(), psi.max(), 30)
cs = ax.contour(Z, R, psi, levels=levels, cmap="RdBu_r")
ax.set_xlabel("z [m]")
ax.set_ylabel("r [m]")
ax.set_title(r"Magnetic flux function $\psi(r,z)$")
plt.colorbar(cs, ax=ax, label=r"$\psi$ [T$\cdot$m$^2$]")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 4. Three definitions of detachment efficiency

Helicon computes three definitions to resolve ambiguity in the literature:

| Definition | Formula | Physical meaning |
|---|---|---|
| **Momentum-based** $\eta_d^{(p)}$ | $p_z^{\text{exit}} / p_z^{\text{inject}}$ | What fraction of injected momentum exits downstream |
| **Particle-based** $\eta_d^{(N)}$ | $N_{\text{exit}} / N_{\text{inject}}$ | What fraction of particles escape |
| **Energy-based** $\eta_d^{(E)}$ | $E_z^{\text{exit}} / E_{\text{inject}}$ | What fraction of kinetic energy is directed axially |

```python
from helicon.postprocess.detachment import compute_detachment

det = compute_detachment("path/to/warpx_output")
print(det.summary())
```

Typical healthy output:
```
Detachment Efficiency:
  eta_d (momentum): 0.7823
  eta_d (particle): 0.8541
  eta_d (energy):   0.7102
  Injected: 1000000
  Exited downstream: 854100
  Lost radially: 102300
  Reflected: 43600
```

**Key insight:** $\eta_d^{(N)} > \eta_d^{(p)} > \eta_d^{(E)}$ is the typical
ordering. Particles can escape but carry less momentum/energy than injected
due to radial velocity components.

## 5. Electron magnetization map

The electron magnetization parameter $\Omega_e = \omega_{ce} / \nu_{\text{eff}}$
determines where electrons decouple from field lines.
Detachment occurs where $\Omega_e \sim 1$.

In [ ]:
from helicon.postprocess.plume import compute_electron_magnetization

# Synthetic electron density for demonstration
n_e = 1e18 * np.exp(-((R / 0.15) ** 2)) * np.exp(-(((Z - 0.5) / 1.0) ** 2))
T_e_eV = 10.0

Omega_e = compute_electron_magnetization(bfield.Br, bfield.Bz, n_e, T_e_eV)

fig, ax = plt.subplots(figsize=(12, 4))
pcm = ax.pcolormesh(
    Z, R, np.log10(np.clip(Omega_e, 1e-3, 1e6)), cmap="coolwarm", shading="auto"
)
ax.contour(Z, R, Omega_e, levels=[1.0], colors="black", linewidths=2)
ax.set_xlabel("z [m]")
ax.set_ylabel("r [m]")
ax.set_title(r"Electron magnetization $\log_{10}(\Omega_e)$ — detachment at $\Omega_e = 1$")
plt.colorbar(pcm, ax=ax, label=r"$\log_{10}(\Omega_e)$")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 6. Pressure anisotropy

Downstream of the throat, the perpendicular and parallel pressures diverge.
The anisotropy $A = P_\perp / P_\parallel - 1$ is a measure of how
far the plasma deviates from local thermodynamic equilibrium.

```python
from helicon.postprocess.plume import compute_pressure_anisotropy

# Using moment data from compute_moments:
A = compute_pressure_anisotropy(moments.pressure_rr, moments.pressure_zz)
```

Large positive $A$ (P_perp >> P_parallel) indicates strong magnetic confinement.
Negative $A$ indicates parallel-dominated heating typical of expanding plasmas.

## 7. Summary report

Helicon aggregates all metrics into a single JSON report:

```python
from helicon.postprocess.report import generate_report, save_report

report = generate_report("path/to/warpx_output")
save_report(report, "results/report.json")
```

The report includes:
- Thrust, Isp, exhaust velocity
- All three detachment efficiencies
- Plume half-angle, beam efficiency, thrust coefficient
- Radial loss fraction
- Helicon version and config hash for reproducibility